# RAW to RGB — Part 3: White Balance and Edge-Directed Debayering

In this notebook we:
1. Apply **white balance** by sampling a neutral grey from the image
2. Implement **edge-directed (gradient-aware) debayering** to reduce zipper artifacts

The key idea of edge-directed debayering: instead of always averaging neighbours in all
four directions, we look at the local gradient and interpolate **along** the edge
(where the signal is smooth) rather than **across** it (where it changes rapidly).

In [ ]:
import numpy as np              # needed for image arrays
import HdM as HdM               # a useful function I prepared for you to display images pixel per pixel on your monitor
from scipy.ndimage import zoom  # Useful function to scale images

# The sRGB function introduced in the "Display Nonlinearity" script. We need this for correct display of linear data on your monitor
linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)


In [ ]:
# Load data
data = np.load('21_RGB_BilinearDebayer.npz')
rgb_lin_int     = data['rgb_lin_int']
raw             = data['raw']
black_noise_std = data['black_noise_std']

print(f'Loaded: rgb_lin_int {rgb_lin_int.shape},  raw {raw.shape}')

## White Balance

Advanced debayering methods require the raw data to be white balanced before debazering.

The camera's sensor responds differently in R, G, B to a neutral grey.
We correct this by **dividing each channel by the value it measures on a neutral patch**.

Sample a grey patch using the cursor in the image below (pick values from a white or
grey card in the scene), then enter them as `mean_white_R/G/B`.

> **Important:** Sample from the *un-gamma-corrected* image (`imshow(rgb)`) so that
> the values you read match the linear domain we are working in.

In [ ]:
# Display the raw (no gamma) image for sampling
print("Linear RGB. It's fine if it looks wrong. Use this image to pick white balance values")
HdM.show( zoom( rgb_lin_int, [0.1,0.1,1] ) )

In [ ]:
# Values sampled from a white patch in the image. Don't forget to devide by 255 beacuse plotly displays 8bit values.
mean_white_r =   # Hinweis: Sample linear red value from the white patch in the image above
mean_white_g =   # Hinweis: Sample linear green value from the white patch in the image above
mean_white_b =   # Hinweis: Sample linear blue value from the white patch in the image above

# These are your measured white values
wb = np.array([mean_white_r, mean_white_g, mean_white_b])

# Divide each channel by the measured wb values and the white patch will become [1.0, 1.0, 1.0] white.
rgb_lin_int_wb = rgb_lin_int / wb

HdM.show( zoom( rgb_lin_int_wb, [0.1,0.1,1] ) )

## Edge-Directed Debayering

Bilinear debayering averages all four neighbours equally, which creates
**zipper artifacts** at sharp edges. The edge-directed approach detects the
local gradient direction and interpolates along the edge instead of across it.

### Green channel interpolation

For the green channel at a missing pixel, we compare the horizontal and vertical gradient:

- $\Delta_H = |G_{left} - G_{right}|$  
- $\Delta_V = |G_{above} - G_{below}|$

Then:
- if $\Delta_H > \Delta_V$: interpolate **vertically** (edge runs left–right)
- if $\Delta_V > \Delta_H$: interpolate **horizontally** (edge runs top–bottom)
- if equal: use the standard bilinear average

### R and B channels — chrominance trick

Instead of interpolating R and B directly, we interpolate the **chrominance** $R-G$ and $B-G$
using the regular bilinear kernels, then add back the (already well-interpolated) green.
This preserves colour accuracy at edges.

In [ ]:
# For debugging start with a very small test sample: Comment out these lines if everything runs smoothly
raw = np.array([[1, 1, 3, 40],[5, 0, 2, 4],[7, 4, 1, 6],[20, 2, 8, 6]]) # Comment me out after debugging
r  = raw[1::2,1::2] # Comment me out after debugging
g0 = raw[0::2,1::2] # Comment me out after debugging
g1 = raw[1::2,0::2] # Comment me out after debugging
b  = raw[0::2,0::2] # Comment me out after debugging

# only if this works umcomment the following lines which load the original raw bayer data



# Initialize the green image plane full of zeros:
g_ed_int = np.zeros((raw.shape[0]-2, raw.shape[1]-2), dtype=np.float32)





# Next reconstruct the green channel by bilinear interploation:
g_lin_int[0::2,0::2] =   # Hinweis: Calculate by interpolation from 'g0 and g1'
g_lin_int[1::2,0::2] =   # Hinweis: Calculate by indexing from 'g0'
g_lin_int[0::2,1::2] =   # Hinweis: Calculate by indexing from 'g0'
g_lin_int[1::2,1::2] =   # Hinweis: Calculate by interpolation from 'g0 and g1'

print("This is your full resolution reconstruction of the green color channel")
HdM.show(linear2sRGB(np.clip(g_lin_int * 2**stops_above_white, 0, 1)))





# Quick unit test — vertical edge: green should interpolate horizontally
test_block = np.array([[1, 0.9, 0.1, 0],
                        [1, 1.0, 0.2, 0],
                        [1, 0.8, 0.1, 0],
                        [1, 0.9, 0.0, 0]], dtype=float)
print('Edge-directed test result:')
print(g_interpolation_edge_directed(test_block))

In [ ]:
# Apply white balance to the raw Bayer plane before debayering
# (WB pattern follows the RGGB Bayer mosaic)
H, W = bayer.shape
wb_cfa = np.empty((H, W))
wb_cfa[0::2, 0::2] = mean_white_R
wb_cfa[0::2, 1::2] = mean_white_G
wb_cfa[1::2, 0::2] = mean_white_G
wb_cfa[1::2, 1::2] = mean_white_B

bayer_wb = bayer / wb_cfa

# ---- Edge-directed green interpolation (block-by-block) ----
# We process every non-overlapping 2×2 block using a 4×4 context window.
# For speed we vectorise using numpy stride tricks instead of Python loops.

pad = np.pad(bayer_wb, 1, mode='reflect')   # 1-pixel border for context

g_edge = np.zeros((H, W))

# Copy known green values first
g_edge[0::2, 1::2] = bayer_wb[0::2, 1::2]  # top-right green
g_edge[1::2, 0::2] = bayer_wb[1::2, 0::2]  # bottom-left green

# Fill missing green values (at R and B positions)
for row in range(0, H, 2):
    for col in range(0, W, 2):
        block = pad[row:row+4, col:col+4]
        result = g_interpolation_edge_directed(block)
        g_edge[row,   col  ] = result[0, 0]   # upper-left  (R position)
        r_end = min(row+1, H-1)
        c_end = min(col+1, W-1)
        if r_end < H and c_end < W:
            g_edge[r_end, c_end] = result[1, 1]   # lower-right (B position)

print('Green channel done.')

In [ ]:
# ---- R and B via chrominance interpolation ----
# Interpolate (R-G) and (B-G) chrominance using bilinear kernels,
# then recover R = (R-G) + G and B = (B-G) + G.


chroma_RG = bayer_wb - g_edge
chroma_BG = bayer_wb - g_edge

# Masks for R and B positions
mask_R = np.zeros((H, W), bool); mask_R[0::2, 0::2] = True
mask_B = np.zeros((H, W), bool); mask_B[1::2, 1::2] = True

# Bilinear kernel (same as notebook 21)
kernel_RB = np.array([[1, 2, 1],
                       [2, 4, 2],
                       [1, 2, 1]]) / 4.0

def np_convolve2d(image, kernel, mode='mirror'):
    """Pure NumPy 2D convolution — replaces scipy.ndimage.convolve."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    pad_mode = 'reflect' if mode == 'mirror' else 'constant'
    img_pad  = np.pad(image, ((ph, ph), (pw, pw)), mode=pad_mode)
    H, W     = image.shape
    out      = np.zeros((H, W))
    for i in range(kh):
        for j in range(kw):
            out += kernel[i, j] * img_pad[i:i+H, j:j+W]
    return out

def bilinear_chroma(chroma, mask, kernel):
    values = chroma * mask
    weight = np_convolve2d(mask.astype(float), kernel, mode='mirror')
    interp = np_convolve2d(values, kernel, mode='mirror') / np.maximum(weight, 1e-6)
    return np.where(mask, chroma, interp)

chroma_RG_interp = bilinear_chroma(chroma_RG, mask_R, kernel_RB)
chroma_BG_interp = bilinear_chroma(chroma_BG, mask_B, kernel_RB)

r_edge = chroma_RG_interp + g_edge
b_edge = chroma_BG_interp + g_edge

rgb_edge = np.stack([r_edge, g_edge, b_edge], axis=2)
print('Edge-directed debayering complete.')

## Compare: bilinear vs. edge-directed

In [ ]:
vo, ho = 1220, 1050
size   = 200

crop_bilinear = rgb[vo*2:(vo+size)*2, ho*2:(ho+size)*2, :] / wb
crop_edge     = rgb_edge[vo*2:(vo+size)*2, ho*2:(ho+size)*2, :]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(linear2sRGB(np.clip(crop_bilinear, 0, 1)))
axes[0].set_title('Bilinear interpolation')
axes[0].axis('off')

axes[1].imshow(linear2sRGB(np.clip(crop_edge, 0, 1)))
axes[1].set_title('Edge-directed interpolation')
axes[1].axis('off')

plt.suptitle('Edge sharpness comparison — zipper artifacts reduced')
plt.tight_layout()
plt.show()

## Save result

In [ ]:
rgb_final = rgb_edge   # use edge-directed result going forward

np.savez('22_RGB_EdgeDirectedDebayer.npz',
         rgb=rgb_final,
         black_noise_std=black_noise_std)

print('Saved: 22_RGB_EdgeDirectedDebayer.npz')